<a id="top"></a>
# Estimating Galaxy Cluster Masses with Convolutional Neural Networks

***

## Imports
This notebook uses the following:
- *numpy* to handle array functions
- *astropy.io fits* for accessing FITS files
- *matplotlib.pyplot* for plotting data
- *keras* for building the CNN

In [ ]:
%matplotlib widget
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
import pandas as pd
import keras
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, Conv2D, MaxPooling2D
from keras.layers import GlobalAveragePooling2D


## Add cluster aware cross-validation

The following cells convert the FITS metadata to a DataFrame, check cluster completeness, and run cluster-aware cross-validation.

In [ ]:
data = {}

table = hdul[2].data

for name in table.names:
    arr = np.array(table[name])

    if arr.dtype.byteorder not in ('=', '|'):
        arr = arr.astype(arr.dtype.newbyteorder('='))

    data[name] = arr

df = pd.DataFrame(data)

### 3. Check cluster completeness

Each complete cluster should contain exactly the three lines of sight: `x`, `y`, and `z`. Incomplete clusters are excluded from cross-validation.

In [ ]:
# Verify that every cluster has exactly the three expected lines of sight.
los_by_cluster = df.groupby('cluster_id')['view_los'].agg(lambda values: set(values))
expected_los = set(df['view_los'].dropna().unique())
complete_clusters = los_by_cluster[los_by_cluster == expected_los].index.to_numpy()
incomplete_clusters = los_by_cluster[los_by_cluster != expected_los]

print(f'Expected lines of sight: {sorted(expected_los)}')
print(f'Complete clusters: {len(complete_clusters)} out of {len(los_by_cluster)}')
print('Incomplete or duplicated clusters:')
print(incomplete_clusters)

assert all(len(df[df['cluster_id'] == cid]) == len(expected_los) for cid in complete_clusters), \
    'At least one cluster has duplicate rows for a line of sight.'

In [ ]:
images = hdul[1].data

### 6. Cluster-aware 10-fold cross-validation

The CV loop splits complete `cluster_id` groups, so all three lines of sight remain in the same fold. Cluster 0 is excluded because it is incomplete. Set `epochs` and uncomment `cv_rot` when the rotation-augmented comparison is needed.

In [ ]:
n_folds = 10
epochs = 50
batch_size = 16

# `clusters` contains one entry per cluster, not one entry per image.
# Therefore KFold assigns complete cluster IDs to folds.
clusters = np.sort(complete_clusters)
assert len(clusters) >= n_folds

# Store every image row belonging to each cluster. These indices are expanded
# only after the cluster-level fold assignment has been made.
cluster_indices = {
    cid: df.index[df['cluster_id'] == cid].to_numpy()
    for cid in clusters
}
cluster_masses = {
    cid: df.loc[df['cluster_id'] == cid, 'log_M500'].iloc[0]
    for cid in clusters
}
norm_cv = np.nanmin(list(cluster_masses.values()))
cluster_masses = {cid: mass - norm_cv for cid, mass in cluster_masses.items()}

In [ ]:
def build_model(with_rotation=False, input_shape=(*images.shape[1:], 1)):
    model_layers = [Input(shape=input_shape)]
    if with_rotation:
        model_layers.extend([
            keras.layers.RandomRotation(0.25),
            keras.layers.RandomFlip('horizontal'),
            keras.layers.RandomFlip('vertical')
        ])
    model_layers.extend([
        Conv2D(16, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(32, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        GlobalAveragePooling2D(),
        Dropout(0.1),
        Dense(200, activation='relu'),
        Dropout(0.1),
        Dense(100, activation='relu'),
        Dense(20, activation='relu'),
        Dense(1, activation='linear')
    ])
    model = Sequential(model_layers)
    model.compile(loss='mean_squared_error',
                  optimizer=keras.optimizers.Adam(learning_rate=0.0002),
                  run_eagerly=True,
                  jit_compile=False)
    return model

# Aproch 3

## Approach 3: Explicit geometric augmentation followed by 10-fold CV

This approach explicitly applies the geometric symmetries of a square image to each original image. We consider the original image, horizontal and vertical flips, and rotations by 90, 180, and 270 degrees. These operations are not all independent: combining a flip with rotations produces the same eight transformations as the dihedral group of the square.

For a generic image, there are at most eight distinct outputs per original image. Therefore, three lines of sight can produce at most `3 x 8 = 24` unique images per cluster. Exact duplicates are removed below, so symmetric images cannot be counted twice.

The folds are assigned using the original cluster IDs before or alongside augmentation. Every augmented copy inherits its source cluster ID and mass, so all copies of a cluster remain in one fold. The CV model uses `with_rotation=False` because the rotations have already been materialized in the dataset; adding Keras `RandomRotation` would apply another, separate augmentation on top of them.

In [ ]:
def geometric_variants(image):
    """Return exact unique D4 variants of one square image."""
    candidates = []
    transform_names = []
    for rotation in range(4):
        rotated = np.rot90(image, rotation)
        for flip_name, candidate in [
            ('none', rotated),
            ('horizontal', np.fliplr(rotated)),
            ('vertical', np.flipud(rotated)),
        ]:
            if not any(np.array_equal(candidate, previous) for previous in candidates):
                candidates.append(candidate.copy())
                transform_names.append(f'rot{rotation * 90}_{flip_name}')
    return candidates, transform_names

# Show the redundancy count for one representative cluster.
example_cluster = clusters[0]
example_indices = cluster_indices[example_cluster]
example_counts = []
for source_index in example_indices:
    variants, names = geometric_variants(images[source_index])
    example_counts.append(len(variants))

print(f'Example cluster {example_cluster}: {len(example_indices)} original images')
print(f'Unique variants per image: {example_counts}')
print(f'Unique augmented images in this cluster: {sum(example_counts)}')
assert all(count <= 8 for count in example_counts)
assert sum(example_counts) <= 24

In [ ]:
# Display all unique variants for the three original images in one cluster.
fig, axes = plt.subplots(3, 8, figsize=(12, 6))
for row, source_index in enumerate(example_indices):
    variants, names = geometric_variants(images[source_index])
    for column, axis in enumerate(axes[row]):
        axis.axis('off')
        if column < len(variants):
            axis.imshow(variants[column], cmap='magma')
            axis.set_title(names[column], fontsize=8)
    axes[row, 0].set_ylabel(f'LOS {row + 1}', rotation=90, fontsize=10)
plt.suptitle(
    f'Unique geometric variants for cluster {example_cluster} '
    '(up to 8 per original image)',
    y=1.02
)
plt.tight_layout()
plt.show()

### Exampl 2

In [ ]:

# Show the redundancy count for one representative cluster.
example_cluster = clusters[5]
example_indices = cluster_indices[example_cluster]
example_counts = []
for source_index in example_indices:
    variants, names = geometric_variants(images[source_index])
    example_counts.append(len(variants))

print(f'Example cluster {example_cluster}: {len(example_indices)} original images')
print(f'Unique variants per image: {example_counts}')
print(f'Unique augmented images in this cluster: {sum(example_counts)}')
assert all(count <= 8 for count in example_counts)
assert sum(example_counts) <= 24

In [ ]:
# Display all unique variants for the three original images in one cluster.
fig, axes = plt.subplots(3, 8, figsize=(12, 6))
for row, source_index in enumerate(example_indices):
    variants, names = geometric_variants(images[source_index])
    for column, axis in enumerate(axes[row]):
        axis.axis('off')
        if column < len(variants):
            axis.imshow(variants[column], cmap='magma')
            axis.set_title(names[column], fontsize=8)
    axes[row, 0].set_ylabel(f'LOS {row + 1}', rotation=90, fontsize=10)
plt.suptitle(
    f'Unique geometric variants for cluster {example_cluster} '
    '(up to 8 per original image)',
    y=1.02
)
plt.tight_layout()
plt.show()

### 3. Build the deduplicated augmented dataset

The dataset is augmented cluster by cluster. The source cluster ID is stored for every generated image, which lets cross-validation split on clusters rather than augmented rows. This gives each complete cluster up to 24 images: three original lines of sight times eight unique square-symmetry variants.

In [ ]:
augmented_images_list = []
augmented_cluster_ids = []
augmented_masses = []
augmented_transform_names = []

for cluster_id in clusters:
    source_indices = cluster_indices[cluster_id]
    cluster_mass = cluster_masses[cluster_id]
    for source_index in source_indices:
        variants, names = geometric_variants(images[source_index])
        augmented_images_list.extend(variants)
        augmented_cluster_ids.extend([cluster_id] * len(variants))
        augmented_masses.extend([cluster_mass] * len(variants))
        augmented_transform_names.extend(names)

augmented_images = np.stack(augmented_images_list)
augmented_cluster_ids = np.asarray(augmented_cluster_ids)
augmented_masses = np.asarray(augmented_masses)

print(f'Original image rows: {len(images)}')
print(f'Complete-cluster image rows: {sum(len(cluster_indices[cid]) for cid in clusters)}')
print(f'Deduplicated augmented image rows: {len(augmented_images)}')
print(f'Augmented images per complete cluster:')
print(pd.Series(augmented_cluster_ids).value_counts().value_counts().sort_index())

assert set(augmented_cluster_ids) == set(clusters)
assert len(augmented_images) == len(augmented_cluster_ids) == len(augmented_masses)
assert all(
    len(augmented_cluster_ids[augmented_cluster_ids == cluster_id]) <= 24
    for cluster_id in clusters
)

### 4. Cluster-level 10-fold CV on the augmented images

The model below uses `build_model(with_rotation=False)`: there is no additional random rotation layer. The explicit rotations and flips are already present in `augmented_images`. Fold assignment is still performed using the original `clusters` array, so no augmented copy of a validation cluster can appear in training.

In [ ]:
import time

augmented_indices_by_cluster = {
    cluster_id: np.flatnonzero(augmented_cluster_ids == cluster_id)
    for cluster_id in clusters
}


def run_augmented_cv():
    folds = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_results_augmented = []
    fold_histories_augmented = []
    total_start = time.perf_counter()
    final_model = None

    print('Starting augmented-data 10-fold cross-validation')
    print(f'Total complete clusters: {len(clusters)}')
    print(f'Total augmented images: {len(augmented_images)}')
    print(f'Configuration: {n_folds} folds, {epochs} epochs per fold, batch size {batch_size}')
    print('-' * 72)

    for fold, (train_ids, val_ids) in enumerate(folds.split(clusters), start=1):
        fold_start = time.perf_counter()
        train_clusters = clusters[train_ids]
        val_clusters = clusters[val_ids]
        train_indices = np.concatenate(
            [augmented_indices_by_cluster[cluster_id] for cluster_id in train_clusters]
        )
        val_indices = np.concatenate(
            [augmented_indices_by_cluster[cluster_id] for cluster_id in val_clusters]
        )

        train_X_augmented = augmented_images[train_indices].reshape(
            -1, *augmented_images.shape[1:], 1
        )
        val_X_augmented = augmented_images[val_indices].reshape(
            -1, *augmented_images.shape[1:], 1
        )
        train_Y_augmented = augmented_masses[train_indices]
        val_Y_augmented = augmented_masses[val_indices]

        print(
            f'Fold {fold}/{n_folds} starting | '
            f'train: {len(train_clusters)} clusters, {len(train_indices)} images | '
            f'validation: {len(val_clusters)} clusters, {len(val_indices)} images'
        )
        print(f'Fold {fold}: training for {epochs} epochs...')

        model = build_model(
            with_rotation=False,
            input_shape=(*augmented_images.shape[1:], 1)
        )
        history = model.fit(
            train_X_augmented,
            train_Y_augmented,
            validation_data=(val_X_augmented, val_Y_augmented),
            batch_size=batch_size,
            epochs=epochs,
            verbose=1
        )
        predictions = model.predict(
            val_X_augmented,
            batch_size=batch_size,
            verbose=0
        ).flatten()
        residuals = val_Y_augmented - predictions
        final_model = model
        fold_histories_augmented.append(history)
        fold_results_augmented.append({
            'fold': fold,
            'r2': r2_score(val_Y_augmented, predictions),
            'mae': mean_absolute_error(val_Y_augmented, predictions),
            'rmse': np.sqrt(mean_squared_error(val_Y_augmented, predictions)),
            'scatter': np.std(residuals),
            'bias': np.mean(residuals),
            'val_loss': history.history['val_loss'][-1],
            'train_images': len(train_indices),
            'val_images': len(val_indices)
        })
        fold_result = fold_results_augmented[-1]
        fold_elapsed = time.perf_counter() - fold_start
        print(
            f"Fold {fold}/{n_folds} complete in {fold_elapsed / 60:.1f} min | "
            f"R2={fold_result['r2']:.4f}, "
            f"MAE={fold_result['mae']:.4f}, "
            f"RMSE={fold_result['rmse']:.4f}, "
            f"val_loss={fold_result['val_loss']:.6f}"
        )
        print(f'Elapsed total: {(time.perf_counter() - total_start) / 60:.1f} min')
        print('-' * 72)

    print(
        f'Augmented 10-fold CV finished in '
        f'{(time.perf_counter() - total_start) / 60:.1f} min'
    )
    return pd.DataFrame(fold_results_augmented), fold_histories_augmented, final_model

In [ ]:
cv_augmented, histories_augmented, final_augmented_model = run_augmented_cv()
print('Approach 3: explicit geometric augmentation + cluster-level 10-fold CV')
display(cv_augmented)
print('Output matrix summary:')
display(cv_augmented[
    ['r2', 'mae', 'rmse', 'scatter', 'bias', 'val_loss']
].agg(['mean', 'std', 'min', 'max']))

# Approach 3 outputs
All models, metrics, histories, plots, and plotting scripts are saved under `approach3_outputs`.


In [ ]:
from pathlib import Path
import json
output_dir = Path('approach3_outputs')
output_dir.mkdir(exist_ok=True)


In [ ]:
final_augmented_model.save(output_dir / 'last_augmented_cv_model.keras')
cv_augmented.to_csv(output_dir / 'cv_augmented.csv', index=False)
np.savez(output_dir / 'training_histories_augmented.npz', train_loss=np.asarray([h.history['loss'] for h in histories_augmented]), val_loss=np.asarray([h.history['val_loss'] for h in histories_augmented]))
(output_dir / 'run_metadata.json').write_text(json.dumps({'approach': 3, 'n_folds': int(n_folds), 'fold_seed': 42, 'augmentation': 'deduplicated D4 transformations'}, indent=2))
print(f'Saved final augmented model, CV table, and histories to {output_dir.resolve()}')

In [ ]:
metrics = ['r2', 'mae', 'rmse', 'scatter', 'bias', 'val_loss']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for axis, metric in zip(axes.flat, metrics):
    axis.bar(cv_augmented['fold'], cv_augmented[metric], color='C3')
    axis.set_title(metric)
    axis.set_xlabel('Fold')
    axis.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(output_dir / 'augmented_cv_metrics_by_fold.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
train_loss = np.asarray([h.history['loss'] for h in histories_augmented])
val_loss = np.asarray([h.history['val_loss'] for h in histories_augmented])
epoch_axis = np.arange(1, train_loss.shape[1] + 1)
fig, axis = plt.subplots(figsize=(10, 6))
for losses, label, color in [(train_loss, 'Training', 'C0'), (val_loss, 'Validation', 'C1')]:
    mean_loss = losses.mean(axis=0)
    std_loss = losses.std(axis=0)
    axis.plot(epoch_axis, mean_loss, color=color, label=f'{label} mean')
    axis.fill_between(epoch_axis, mean_loss - std_loss, mean_loss + std_loss, color=color, alpha=0.18)
axis.set(xlabel='Epoch', ylabel='MSE', title='Augmented CV loss curves')
axis.legend()
fig.tight_layout()
fig.savefig(output_dir / 'augmented_cv_loss_curves.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved augmented CV plots')


In [ ]:
plot_script = r'''from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path(__file__).resolve().parent
results = pd.read_csv(OUTPUT_DIR / 'cv_augmented.csv')
histories = np.load(OUTPUT_DIR / 'training_histories_augmented.npz')
metrics = ['r2', 'mae', 'rmse', 'scatter', 'bias', 'val_loss']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for axis, metric in zip(axes.flat, metrics):
    axis.bar(results['fold'], results[metric], color='C3')
    axis.set_title(metric)
    axis.set_xlabel('Fold')
    axis.grid(axis='y', alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'augmented_cv_metrics_by_fold.png', dpi=200, bbox_inches='tight')
plt.close(fig)

fig, axis = plt.subplots(figsize=(10, 6))
for key, label, color in [('train_loss', 'Training', 'C0'), ('val_loss', 'Validation', 'C1')]:
    losses = histories[key]
    epoch_axis = np.arange(1, losses.shape[1] + 1)
    mean_loss = losses.mean(axis=0)
    std_loss = losses.std(axis=0)
    axis.plot(epoch_axis, mean_loss, color=color, label=f'{label} mean')
    axis.fill_between(epoch_axis, mean_loss - std_loss, mean_loss + std_loss, color=color, alpha=0.18)
axis.set(xlabel='Epoch', ylabel='MSE', title='Augmented CV loss curves')
axis.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'augmented_cv_loss_curves.png', dpi=200, bbox_inches='tight')
plt.close(fig)
print(f'Plots regenerated in {OUTPUT_DIR}')
'''
(output_dir / 'plot_outputs.py').write_text(plot_script)
print(f'Saved plotting script to {(output_dir / "plot_outputs.py").resolve()}')